## step 8 _ Recommendation Engine
#### level 1 : popularty
* 1- Product Popularity per Segment
* 2- Recommendation Function
#### 🔥 Level 2: Global Popularity
#### 🔥 Level 3: Personal History
#### Level 4: Hybrid System

In [1]:
import pandas as pd

In [2]:
rfm_table = pd.read_csv('rfm_table.csv')
rfm_table.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,325,1,77183.60
1,12347,1,7,4310.00
2,12348,74,4,1797.24
3,12349,18,1,1757.55
4,12350,309,1,334.40


In [13]:
sales_data = pd.read_csv('rfm_data.csv')
sales_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392692 entries, 0 to 392691
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    392692 non-null  int64  
 1   StockCode    392692 non-null  object 
 2   Description  392692 non-null  object 
 3   Quantity     392692 non-null  int64  
 4   InvoiceDate  392692 non-null  object 
 5   UnitPrice    392692 non-null  float64
 6   CustomerID   392692 non-null  float64
 7   Country      392692 non-null  object 
dtypes: float64(2), int64(2), object(4)
memory usage: 24.0+ MB


In [3]:
rfm_segment = pd.read_csv('rfm_table_with_segments.csv')
rfm_segment.head()

,Recency,Frequency,Monetary,Segment
0,325,1,77183.60,Active valuable
1,1,7,4310.00,Occasional
2,74,4,1797.24,Occasional
3,18,1,1757.55,Occasional
4,309,1,334.40,Churned


In [12]:
rfm_to_recommend = rfm_segment.copy()
rfm_to_recommend['CustomerID'] = rfm_table['CustomerID']
rfm_to_recommend.head()

,CustomerID,Recency,Frequency,Monetary,Segment
0,12346,325,1,77183.60,Active valuable
1,12347,1,7,4310.00,Occasional
2,12348,74,4,1797.24,Occasional
3,12349,18,1,1757.55,Occasional
4,12350,309,1,334.40,Churned


* 1- Product Popularity per Segment

In [14]:
df_recommend = sales_data.merge(
    rfm_to_recommend[['CustomerID', 'Segment']],
    on='CustomerID',
    how='inner'
)

df_recommend.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Segment
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,Active valuable
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Active valuable
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,Active valuable
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Active valuable
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Active valuable


In [ ]:
#Building the behavior of each segment
segment_products = df_recommend.groupby(
    ['Segment', 'Description']
)['Quantity'].sum().reset_index()

In [16]:
segment_products

,Segment,Description,Quantity
0,Active valuable,4 PURPLE FLOCK DINNER CANDLES,36
1,Active valuable,50'S CHRISTMAS GIFT BAG LARGE,638
2,Active valuable,DOLLY GIRL BEAKER,258
3,Active valuable,I LOVE LONDON MINI BACKPACK,35
4,Active valuable,NINE DRAWER OFFICE TIDY,1
...,...,...,...
12750,VIP,ZINC SWEETHEART WIRE LETTER RACK,11
12751,VIP,ZINC T-LIGHT HOLDER STAR LARGE,139
12752,VIP,ZINC T-LIGHT HOLDER STARS SMALL,557
12753,VIP,ZINC WILLIE WINKIE CANDLE STICK,465


In [ ]:
# Product ranking within each segment
segment_products = segment_products.sort_values(
    ['Segment', 'Quantity'],
    ascending=False
)

* 2- Recommendation Function

In [18]:
def recommend(segment, n=5):
    return segment_products[
        segment_products['Segment'] == segment
    ].head(n)

In [19]:
recommend("Occasional")

,Segment,Description,Quantity
9755,Occasional,WORLD WAR 2 GLIDERS ASSTD DESIGNS,22097
7812,Occasional,JUMBO BAG RED RETROSPOT,18184
6345,Occasional,ASSORTED COLOUR BIRD ORNAMENT,16719
8299,Occasional,PACK OF 72 RETROSPOT CAKE CASES,16712
9671,Occasional,WHITE HANGING HEART T-LIGHT HOLDER,13482


#### 🔥 Level 2: Global Popularity

In [20]:
global_top = sales_data.groupby('Description')['Quantity'].sum().sort_values(ascending=False)
global_top

Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54319
JUMBO BAG RED RETROSPOT               46078
WHITE HANGING HEART T-LIGHT HOLDER    36706
                                      ...  
WHITE ROSEBUD  PEARL EARRINGS             1
WHITE STONE/CRYSTAL EARRINGS              1
FLOWER SHOP DESIGN MUG                    1
ORANGE FELT VASE + FLOWERS                1
 I LOVE LONDON MINI RUCKSACK              1
Name: Quantity, Length: 3877, dtype: int64

#### 🔥 Level 3: Personal History

In [21]:
def user_history(customer_id):
    return sales_data[
        sales_data['CustomerID'] == customer_id
    ]['Description'].value_counts().head(5)

#### Level 4: Hybrid System

In [22]:
def recommend_final(customer_id, segment):
    seg = recommend(segment)
    pop = global_top.head(5)
    hist = user_history(customer_id)

    return {
        "segment_based": seg,
        "popular": pop,
        "personal": hist
    }